# lora_variant_timing

Measures seconds per optimizer step and peak VRAM for four settings of `gradient_checkpointing` and `batch_size`, all holding `global_batch_size: 16`.

| variant | grad ckpt | batch | accum | global |
|---|---|---|---|---|
| A (shipped default) | on | 2 | 8 | 16 |
| B | off | 2 | 8 | 16 |
| C | on | 4 | 4 | 16 |
| D | off | 4 | 4 | 16 |

Both settings are memory-versus-speed tradeoffs. Gradient checkpointing recomputes activations rather than storing them; batch/accum splits the same global batch differently. Neither changes the gradient, the update, or the resulting weights. Section 7 checks that empirically by comparing losses between A and B, which see identical data in identical order.

Measured baseline from `lora_timing_test`: 14.65 s/step, 45.5 GB peak, giving 16.3 h per rank at 4000 steps.

12 steps per variant, roughly 20 minutes and 5 compute units. A100 runtime. Writes nothing to the thesis Drive folders.

## 1. Repo, deps

In [ ]:
import os, subprocess, sys, json

REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/VectorSpaceLab/OmniGen2.git", REPO], check=True)
    print("cloned")
else:
    print("repo already present")

# requirements.txt omits torchdiffeq (omnigen2/transport/integrators.py)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchdiffeq"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)

# peft raises rather than returning False when torchao is older than 0.16.0; Colab ships 0.10.0.
# torchao is unused by the bf16 LoRA training path.
_ao = subprocess.run([sys.executable, "-c",
                      "from peft.import_utils import is_torchao_available; is_torchao_available()"],
                     capture_output=True, text=True)
if _ao.returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("torchao removed")

_probe = ("from peft import LoraConfig;"
          "from peft.import_utils import is_torchao_available; is_torchao_available();"
          "from omnigen2.transport import create_transport;"
          "from omnigen2.dataset.omnigen2_train_dataset import OmniGen2TrainDataset;"
          "print('train.py imports OK')")
chk = subprocess.run([sys.executable, "-c", _probe], cwd=REPO, capture_output=True, text=True)
print(chk.stdout.strip() or chk.stderr.strip()[-1000:])
assert chk.returncode == 0, "train.py imports are broken"

sys.path.insert(0, REPO)
os.chdir(REPO)

import torch
print("gpu  :", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

cloned
torchao removed
train.py imports OK
gpu  : NVIDIA A100-SXM4-80GB | 85.1 GB


## 2. Drive, images, splits

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
IMGS, EXPECTED = "/content/imgs", 37063

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{DRIVE}/imgs_subset.tar" /content/imgs_subset.tar '
                   f'&& tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
print("images:", n_files(IMGS))

for f in ["train.jsonl", "screen_exclude_from_training.txt"]:
    if not os.path.exists(f"/content/{f}"):
        subprocess.run(["cp", f"{DRIVE}/{f}", "/content/"], check=True)
print("splits copied")

Mounted at /content/drive
images: 37063
splits copied


## 3. Timing data

400 rows, enough that no variant repeats within 12 steps x 16 samples. Screen targets excluded.

In [ ]:
DATA = "/content/lora_data"
os.makedirs(f"{DATA}/jsonls", exist_ok=True)

excl = set(open("/content/screen_exclude_from_training.txt").read().split())
train = [json.loads(l) for l in open("/content/train.jsonl", encoding="utf-8")]
kept = [r for r in train if r["output_image"] not in excl]
assert len(kept) == 39591, f"expected 39591, got {len(kept)}"
print(f"train {len(train)} -> {len(kept)} after removing {len(excl)} screen targets")

rows = [{"task_type": "ic",
         "instruction": r["instruction"],
         "input_images": [os.path.join(IMGS, p) for p in r["input_images"]],
         "output_image": os.path.join(IMGS, r["output_image"])}
        for r in kept[:400]]
missing = [p for r in rows for p in r["input_images"] + [r["output_image"]] if not os.path.exists(p)]
assert not missing, f"{len(missing)} missing images"

with open(f"{DATA}/jsonls/timing.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

open(f"{DATA}/ic.yml", "w").write(
    'data:\n  - \n    path: "%s/jsonls/timing.jsonl"\n    type: "ic"\n    ratio: !!float 1\n' % DATA)
open(f"{DATA}/mix.yml", "w").write(
    "data:\n  - \n    path: '%s/ic.yml'\n    type: 'ic'\n    ratio: !!float 1\n" % DATA)
print(f"wrote {len(rows)} rows")

train 39893 -> 39591 after removing 62 screen targets
wrote 400 rows


## 4. Patch

Logs wall time, peak VRAM and loss per optimizer step. Idempotent.

In [ ]:
p = f"{REPO}/train.py"
src = open(p, encoding="utf-8").read()

ANCHOR = "                global_step += 1\n"
PATCH = '''                if accelerator.is_main_process:
                    import time as _t
                    _g = globals().setdefault("_TIMING", {"prev": None})
                    _now = _t.time()
                    if _g["prev"] is not None:
                        print("[timing] step %d %.3fs peak_vram %.2fGB loss %.5f" % (
                            global_step, _now - _g["prev"],
                            torch.cuda.max_memory_allocated() / 1e9,
                            logs.get("loss", float("nan"))), flush=True)
                    _g["prev"] = _now
'''

if "[timing]" in src:
    print("already patched")
else:
    assert src.count(ANCHOR) == 1, f"anchor found {src.count(ANCHOR)} times"
    open(p, "w", encoding="utf-8").write(src.replace(ANCHOR, ANCHOR + PATCH))
    import ast; ast.parse(open(p, encoding="utf-8").read())
    print("patched and parses")

patched and parses


## 5. Run the four variants

Separate process each, so CUDA state and peak-memory counters start clean. An OOM is caught and reported rather than stopping the sweep.

In [ ]:
from omegaconf import OmegaConf
import time, re

STEPS = 12
VARIANTS = [
    ("A_default",  True,  2, 8),
    ("B_nockpt",   False, 2, 8),
    ("C_bs4",      True,  4, 4),
    ("D_nockpt_bs4", False, 4, 4),
]

def build_cfg(name, gc, bs, accum):
    c = OmegaConf.load(f"{REPO}/options/ft_lora.yml")
    c.name = f"variant_{name}"
    c.data.data_path = f"{DATA}/mix.yml"
    c.data.max_output_pixels = 262144            # 512 * 512
    c.data.max_input_pixels = [262144] * 4
    c.data.max_side_length = 512
    c.train.gradient_checkpointing = gc
    c.train.batch_size = bs
    c.train.gradient_accumulation_steps = accum
    c.train.max_train_steps = STEPS
    c.logger.checkpointing_steps = 10 ** 9
    c.logger.log_with = ["tensorboard"]
    c.val.train_visualization_steps = 10 ** 9
    assert bs * accum == c.train.global_batch_size, "global batch must stay 16"
    path = f"{REPO}/options/variant_{name}.yml"
    OmegaConf.save(c, path)
    return path

LAUNCH = ["accelerate", "launch",
          "--num_processes=1", "--num_machines=1", "--machine_rank=0", "--main_process_port=29500",
          "--use_fsdp", "--fsdp_offload_params", "false",
          "--fsdp_sharding_strategy", "HYBRID_SHARD_ZERO2",
          "--fsdp_auto_wrap_policy", "TRANSFORMER_BASED_WRAP",
          "--fsdp_transformer_layer_cls_to_wrap", "OmniGen2TransformerBlock",
          "--fsdp_state_dict_type", "FULL_STATE_DICT",
          "--fsdp_forward_prefetch", "false",
          "--fsdp_use_orig_params", "True",
          "--fsdp_cpu_ram_efficient_loading", "false",
          "--fsdp_sync_module_states", "True"]

results = {}
for name, gc, bs, accum in VARIANTS:
    cfg = build_cfg(name, gc, bs, accum)
    print(f"\n{'='*70}\n{name}: grad_ckpt={gc} batch={bs} accum={accum}\n{'='*70}", flush=True)
    t0 = time.time()
    proc = subprocess.Popen(LAUNCH + ["train.py", "--config", cfg], cwd=REPO,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env={**os.environ, "WANDB_DISABLED": "true"})
    log = []
    for line in proc.stdout:
        log.append(line)
        if "[timing]" in line:
            print(line.rstrip(), flush=True)
    proc.wait()
    txt = "".join(log)
    open(f"/content/log_{name}.txt", "w").write(txt)

    steps = [(int(a), float(b), float(c), float(d)) for a, b, c, d in re.findall(
        r"\[timing\] step (\d+) ([\d.]+)s peak_vram ([\d.]+)GB loss ([\d.nan]+)", txt)]
    oom = "CUDA out of memory" in txt or "OutOfMemoryError" in txt
    results[name] = {"gc": gc, "bs": bs, "accum": accum, "steps": steps,
                     "oom": oom, "rc": proc.returncode, "wall": time.time() - t0}
    if not steps:
        print(f"  FAILED rc={proc.returncode}" + (" (OOM)" if oom else ""))
        print("  " + "\n  ".join(log[-12:]))

print("\nsweep complete")


A_default: grad_ckpt=True batch=2 accum=8
[timing] step 2 16.588s peak_vram 45.22GB loss 0.30502
[timing] step 3 16.597s peak_vram 45.22GB loss 0.38981
[timing] step 4 13.492s peak_vram 45.22GB loss 0.33704
[timing] step 5 16.096s peak_vram 45.22GB loss 0.40411
[timing] step 6 15.756s peak_vram 45.22GB loss 0.39708
[timing] step 7 14.972s peak_vram 45.22GB loss 0.33490
[timing] step 8 14.979s peak_vram 45.22GB loss 0.42004
[timing] step 9 17.283s peak_vram 45.44GB loss 0.32328
[timing] step 10 14.996s peak_vram 45.44GB loss 0.29802
[timing] step 11 13.230s peak_vram 45.44GB loss 0.33010
[timing] step 12 13.998s peak_vram 45.44GB loss 0.29452

B_nockpt: grad_ckpt=False batch=2 accum=8
  FAILED rc=1 (OOM)
  Failures:

    <NO_OTHER_FAILURES>

  ------------------------------------------------------------

  Root Cause (first observed failure):

  [0]:

    time      : 2026-07-23_12:01:30

    host      : e0b437cf2983

    rank      : 0 (local_rank: 0)

    exitcode  : 1 (pid: 7376)

   

## 6. Results

In [ ]:
import statistics as st

BASE = 14.65   # measured in lora_timing_test, variant A settings

print(f"{'variant':14s} {'ckpt':>5s} {'bs':>3s} {'acc':>4s} {'s/step':>8s} {'vs A':>7s} {'peak GB':>8s} {'status':>10s}")
print("-" * 68)

summary = {}
for name, r in results.items():
    if not r["steps"]:
        print(f"{name:14s} {str(r['gc']):>5s} {r['bs']:>3d} {r['accum']:>4d} "
              f"{'-':>8s} {'-':>7s} {'-':>8s} {'OOM' if r['oom'] else 'FAILED':>10s}")
        continue
    warm = [dt for s, dt, _, _ in r["steps"] if s > r["steps"][0][0] + 1]
    med = st.median(warm)
    peak = max(v for _, _, v, _ in r["steps"])
    summary[name] = {"s_step": med, "peak": peak,
                     "loss": [l for _, _, _, l in r["steps"]]}
    ref = summary.get("A_default", {}).get("s_step", med)
    print(f"{name:14s} {str(r['gc']):>5s} {r['bs']:>3d} {r['accum']:>4d} "
          f"{med:>8.2f} {med/ref:>6.2f}x {peak:>8.2f} {'ok':>10s}")

if "A_default" in summary:
    print(f"\nvariant A this session {summary['A_default']['s_step']:.2f}s "
          f"vs {BASE:.2f}s previously - "
          f"{'consistent' if abs(summary['A_default']['s_step']-BASE)/BASE < 0.12 else 'DIFFERS, treat with caution'}")

free = torch.cuda.get_device_properties(0).total_memory / 1e9
for name, s in summary.items():
    if s["peak"] > 0.9 * free:
        print(f"WARNING {name}: peak {s['peak']:.1f} GB is within 10% of {free:.0f} GB")

variant         ckpt  bs  acc   s/step    vs A  peak GB     status
--------------------------------------------------------------------
A_default       True   2    8    14.98   1.00x    45.44         ok
B_nockpt       False   2    8        -       -        -        OOM
C_bs4           True   4    4    15.01   1.00x    51.47         ok
D_nockpt_bs4   False   4    4        -       -        -        OOM

variant A this session 14.98s vs 14.65s previously - consistent


## 7. Math-neutrality check

A and B differ only in gradient checkpointing: same seed, same batch composition, same data order. Their per-step losses should agree to numerical tolerance. C and D reorganise the batch, so their losses legitimately differ and are not compared here.

In [ ]:
if "A_default" in summary and "B_nockpt" in summary:
    la, lb = summary["A_default"]["loss"], summary["B_nockpt"]["loss"]
    n = min(len(la), len(lb))
    d = [abs(x - y) for x, y in zip(la[:n], lb[:n])]
    rel = [abs(x - y) / max(abs(x), 1e-9) for x, y in zip(la[:n], lb[:n])]
    print(f"{'step':>6s} {'A loss':>10s} {'B loss':>10s} {'abs diff':>10s}")
    for i in range(n):
        print(f"{i+2:>6d} {la[i]:>10.5f} {lb[i]:>10.5f} {d[i]:>10.2e}")
    print(f"\nmax abs diff {max(d):.2e} | max rel diff {max(rel):.2e}")
    print("VERDICT:", "identical to numerical tolerance - gradient checkpointing is math-neutral"
          if max(rel) < 1e-3 else
          "LARGER THAN EXPECTED - inspect before relying on this change")
else:
    print("A or B did not complete; cannot compare")

A or B did not complete; cannot compare


## 8. Budget

Three ranks at the full 4000 steps, per-rank checkpoint selection on val loss.

Set `UNITS_PER_H` from observed consumption: units spent divided by session hours. The A100 list rate is about 11.8/h, but sessions bill connected time, so the effective rate is usually higher.

In [ ]:
UNITS_PER_H = 11.77          # compute-unit consumption rate used for cost estimates
EUR_PER_UNIT = 50.28 / 500
HAVE = 150.86

GEN_S = 23.2                 # measured seconds per generated image
N_RANKS, STEPS_FULL = 3, 4000
SCREEN_RUNS, SCREEN_N = 6, 302   # 3 rank comparisons + 3 steps-curve points
TEST_N = 488

eval_h = (SCREEN_RUNS * SCREEN_N + TEST_N) * GEN_S / 3600 + 0.4

print(f"{'variant':14s} {'s/step':>8s} {'h/rank':>8s} {'h x3':>8s} {'total h':>9s} {'units':>8s} {'EUR':>8s} {'verdict':>26s}")
print("-" * 96)
for name in [n for n, _, _, _ in VARIANTS if n in summary]:
    s = summary[name]["s_step"]
    h1 = STEPS_FULL * s / 3600
    tot = N_RANKS * h1 + eval_h
    u = tot * UNITS_PER_H
    need = u - HAVE
    packs = max(0, -(-need // 500)) if need > 0 else 0
    verdict = "fits" if need <= 0 else f"need {need:.0f} more ({packs:.0f} x 500-pack)"
    print(f"{name:14s} {s:>8.2f} {h1:>8.1f} {N_RANKS*h1:>8.1f} {tot:>9.1f} {u:>8.0f} "
          f"{u*EUR_PER_UNIT:>8.2f} {verdict:>26s}")

print(f"\nevaluation is a fixed {eval_h:.1f} h ({eval_h*UNITS_PER_H:.0f} units) in every row")
print(f"available now: {HAVE:.2f} units; 6 Aug adds 600")

if "A_default" in summary:
    a = summary["A_default"]["s_step"]
    best = min(summary.items(), key=lambda kv: kv[1]["s_step"])
    saved_h = N_RANKS * STEPS_FULL * (a - best[1]["s_step"]) / 3600
    print(f"\nfastest variant: {best[0]} at {best[1]['s_step']:.2f}s/step, "
          f"peak {best[1]['peak']:.1f} GB")
    print(f"saving vs default across 3 ranks: {saved_h:.1f} h, "
          f"{saved_h*UNITS_PER_H:.0f} units, {saved_h*UNITS_PER_H*EUR_PER_UNIT:.2f} EUR")
    print(f"applied to QLoRA and soft prompting as well, roughly "
          f"{2*saved_h*UNITS_PER_H:.0f} units across the remaining thesis")

variant          s/step   h/rank     h x3   total h    units      EUR                    verdict
------------------------------------------------------------------------------------------------
A_default         14.98     16.6     49.9      65.2      767    77.11 need 616 more (2 x 500-pack)
C_bs4             15.01     16.7     50.0      65.2      768    77.22 need 617 more (2 x 500-pack)

evaluation is a fixed 15.2 h (179 units) in every row
available now: 150.86 units; 6 Aug adds 600

fastest variant: A_default at 14.98s/step, peak 45.4 GB
saving vs default across 3 ranks: 0.0 h, 0 units, 0.00 EUR
applied to QLoRA and soft prompting as well, roughly 0 units across the remaining thesis


## 9. Revert

In [ ]:
print(subprocess.run(["git", "checkout", "--", "train.py"], cwd=REPO,
                     capture_output=True, text=True).stderr or "train.py reverted")
print("variant configs kept at options/variant_*.yml")

train.py reverted
variant configs kept at options/variant_*.yml
